In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [14]:
# Fix torchao version conflict that breaks peft's LoRA dispatcher on Kaggle
!pip install -q -U "torchao>=0.16.0" 2>/dev/null || pip uninstall -y torchao -q
!pip install -q -U transformers peft datasets

# If torchao still conflicts, this env var tells peft to skip the torchao dispatcher
import os
os.environ["PEFT_NO_TORCHAO"] = "1"

import torch
import numpy as np
import pandas as pd
from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                          TrainingArguments, Trainer)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
print(f"Train: {train.shape}")

OPTIONS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}


Train: (2000, 8)


In [3]:
train['label'] = train['answer'].map(LABEL_MAP)

answer_q1 = train.iloc[150]['label']
print(f"Q1 Answer: {answer_q1}  (answer '{train.iloc[150]['answer']}')")


Q1 Answer: 2  (answer 'C')


In [4]:
row0 = train.iloc[0]
option_b_input = str(row0['prompt']) + " [SEP] " + str(row0['B'])

answer_q2 = len(option_b_input)
print(f"Q2 Answer: {answer_q2}")


Q2 Answer: 407


In [5]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def format_choices(row):
    """Create 5 (prompt [SEP] option) strings for one question."""
    return [str(row['prompt']) + " [SEP] " + str(row[o]) for o in OPTIONS]

# Tokenize row 0's 5 choices
choices_0 = format_choices(row0)
enc_0 = tokenizer(choices_0, padding='max_length', truncation=True,
                  max_length=128, return_tensors='pt')

# Reshape to [1, 5, 128]  (batch, num_choices, seq_len)
input_ids_0 = enc_0['input_ids'].unsqueeze(0)   # (1, 5, 128)
print(f"Shape: {tuple(input_ids_0.shape)}")

answer_q3 = input_ids_0.shape[1]
print(f"Q3 Answer: {answer_q3}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Shape: (1, 5, 128)
Q3 Answer: 5


In [6]:
batch_16 = train.head(16)

all_choices = []
for _, row in batch_16.iterrows():
    all_choices.extend(format_choices(row))   # 16*5 = 80 strings

enc_16 = tokenizer(all_choices, padding='max_length', truncation=True,
                   max_length=128, return_tensors='pt')

# Reshape to [16, 5, 128]
input_ids_16 = enc_16['input_ids'].view(16, 5, 128)
print(f"Shape: {tuple(input_ids_16.shape)}")

answer_q4 = input_ids_16.numel()
print(f"Q4 Answer: {answer_q4}  (= 16 × 5 × 128)")


Shape: (16, 5, 128)
Q4 Answer: 10240  (= 16 × 5 × 128)


In [7]:
mc_model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
mc_model.eval()

# Prepare row 0 as (1, 5, 128)
inputs_0 = {
    'input_ids': enc_0['input_ids'].unsqueeze(0),
    'attention_mask': enc_0['attention_mask'].unsqueeze(0),
    'token_type_ids': enc_0['token_type_ids'].unsqueeze(0),
}

with torch.no_grad():
    out_0 = mc_model(**inputs_0)

print(f"Logits shape: {tuple(out_0.logits.shape)}")

answer_q5 = out_0.logits.shape[1]
print(f"Q5 Answer: {answer_q5}")


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Logits shape: (1, 5)
Q5 Answer: 5


In [8]:
label_0 = torch.tensor([train.iloc[0]['label']])   # (1,)

with torch.no_grad():
    out_loss = mc_model(**inputs_0, labels=label_0)

print(f"Loss: {out_loss.loss}")
print(f"Loss shape: {tuple(out_loss.loss.shape)}")

answer_q6 = out_loss.loss.dim()
print(f"Q6 Answer: {answer_q6}  (scalar → 0 dimensions)")


Loss: 1.6230076551437378
Loss shape: ()
Q6 Answer: 0  (scalar → 0 dimensions)


In [15]:
# Fresh model for LoRA
lora_base = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

lora_model = get_peft_model(lora_base, lora_config)

answer_q7 = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Q7 Answer: {answer_q7}")
lora_model.print_trainable_parameters()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

Q7 Answer: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [10]:
def tokenize_mcq(row, max_length=128):
    choices = [str(row['prompt']) + " [SEP] " + str(row[o]) for o in OPTIONS]
    enc = tokenizer(choices, padding='max_length', truncation=True,
                    max_length=max_length, return_tensors='pt')
    return {
        'input_ids': enc['input_ids'],            # (5, 128)
        'attention_mask': enc['attention_mask'],  # (5, 128)
        'labels': row['label']
    }

first_100 = train.head(100)
records = [tokenize_mcq(row) for _, row in first_100.iterrows()]

hf_dataset = Dataset.from_list([
    {'input_ids': r['input_ids'].tolist(),
     'attention_mask': r['attention_mask'].tolist(),
     'labels': r['labels']}
    for r in records
])

first_item_ids = torch.tensor(hf_dataset[0]['input_ids'])
print(f"First item input_ids shape: {tuple(first_item_ids.shape)}")

answer_q8 = first_item_ids.shape[0]
print(f"Q8 Answer: {answer_q8}  (5 tokenized choices)")


First item input_ids shape: (5, 128)
Q8 Answer: 5  (5 tokenized choices)


In [18]:
from datasets import Dataset
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import get_peft_model
import torch

# Change this if your correct-answer column has a different name
ANSWER_COL = "answer"

# A -> 0, B -> 1, C -> 2, D -> 3, E -> 4
option_to_idx = {option: i for i, option in enumerate(OPTIONS)}

def tokenize_mcq_64(row):
    choices = [
        str(row["prompt"]) + " [SEP] " + str(row[o])
        for o in OPTIONS
    ]

    enc = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    # Convert answer to integer class index
    answer = row[ANSWER_COL]

    if isinstance(answer, str):
        label = option_to_idx[answer]
    else:
        label = int(answer)

    return {
        "input_ids": enc["input_ids"].tolist(),
        "attention_mask": enc["attention_mask"].tolist(),
        "labels": label
    }


# Tokenize first 32 rows
train_32 = Dataset.from_list([
    tokenize_mcq_64(row)
    for _, row in train.head(32).iterrows()
])


# Fresh LoRA model
ft_base = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

ft_model = get_peft_model(
    ft_base,
    lora_config
)


training_args = TrainingArguments(
    output_dir="/kaggle/working/lora_mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    remove_unused_columns=False,
)


def collate_fn(features):
    return {
        "input_ids": torch.tensor(
            [f["input_ids"] for f in features],
            dtype=torch.long
        ),
        "attention_mask": torch.tensor(
            [f["attention_mask"] for f in features],
            dtype=torch.long
        ),
        "labels": torch.tensor(
            [f["labels"] for f in features],
            dtype=torch.long
        ),
    }


trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_32,
    data_collator=collate_fn,
)


train_result = trainer.train()

answer_q9 = train_result.global_step

print(f"\nQ9 Answer: {answer_q9}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch

Step,Training Loss
1,3.382658
2,3.247322
3,3.115119
4,3.247737



Q9 Answer: 4


In [19]:
ft_model.eval()

# Tokenize row 0 at max_length=64 (same as fine-tuning)
choices_0_64 = [str(row0['prompt']) + " [SEP] " + str(row0[o]) for o in OPTIONS]
enc_0_64 = tokenizer(choices_0_64, padding='max_length', truncation=True,
                     max_length=64, return_tensors='pt')

inputs_0_64 = {
    'input_ids': enc_0_64['input_ids'].unsqueeze(0).to(ft_model.device),
    'attention_mask': enc_0_64['attention_mask'].unsqueeze(0).to(ft_model.device),
    'token_type_ids': enc_0_64['token_type_ids'].unsqueeze(0).to(ft_model.device),
}

with torch.no_grad():
    logits_0 = ft_model(**inputs_0_64).logits   # (1, 5)

probs_0 = torch.softmax(logits_0, dim=1)[0]
print(f"Probabilities A-E: {[round(p,4) for p in probs_0.tolist()]}")

answer_q10 = round(probs_0[4].item(), 4)   # Option E = index 4
print(f"\nQ10 Answer (Option E): {answer_q10}")


Probabilities A-E: [0.2034, 0.2053, 0.2041, 0.1948, 0.1924]

Q10 Answer (Option E): 0.1924


In [20]:
print("=" * 55)
print("  MILESTONE 4 — ANSWER SUMMARY")
print("=" * 55)
print(f"  Q1  Label at index 150:              {answer_q1}")
print(f"  Q2  Char length (row 0, Option B):   {answer_q2}")
print(f"  Q3  Second dimension (num choices):  {answer_q3}")
print(f"  Q4  Total token positions:           {answer_q4}")
print(f"  Q5  Logits per question:             {answer_q5}")
print(f"  Q6  Loss tensor dimensions:          {answer_q6}")
print(f"  Q7  LoRA trainable parameters:       {answer_q7}")
print(f"  Q8  Tokenized choices in input_ids:  {answer_q8}")
print(f"  Q9  Final global_step:               {answer_q9}")
print(f"  Q10 Probability Option E:            {answer_q10}")
print("=" * 55)


  MILESTONE 4 — ANSWER SUMMARY
  Q1  Label at index 150:              2
  Q2  Char length (row 0, Option B):   407
  Q3  Second dimension (num choices):  5
  Q4  Total token positions:           10240
  Q5  Logits per question:             5
  Q6  Loss tensor dimensions:          0
  Q7  LoRA trainable parameters:       295681
  Q8  Tokenized choices in input_ids:  5
  Q9  Final global_step:               4
  Q10 Probability Option E:            0.1924
